In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

# --- 1. Load the ORIGINAL training data ---
try:
    train_df = pd.read_csv('data/train.csv')
except FileNotFoundError:
    print("ERROR: Please place the original 'train.csv' file in your 'data' folder.")

# --- 2. Define all your preprocessing functions ---
def extract_ipq(text):
    text = str(text).lower()
    patterns = [r'pack of (\d+)', r'(\d+)\s*count', r'(\d+)\s*per case', r'pk-\s*(\d+)']
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return int(match.group(1))
    return 1

def extract_weight_volume(text):
    text = str(text).lower()
    value_match = re.search(r'value:\s*([\d\.]+)', text)
    unit_match = re.search(r'unit:\s*(\w+)', text)
    if value_match and unit_match:
        try:
            value = float(value_match.group(1))
            unit = unit_match.group(1)
            if unit in ['ounce', 'oz', 'fl oz']: return value
            if unit in ['pound', 'lb']: return value * 16
            return value
        except ValueError: pass
    patterns = {
        'ounce': r'(\d*\.?\d+)\s*(ounce|oz|fl oz)',
        'pound': r'(\d*\.?\d+)\s*(pound|lb)',
        'gram': r'(\d*\.?\d+)\s*(g|gram)'
    }
    for unit_type, pattern in patterns.items():
        match = re.search(pattern, text)
        if match:
            try:
                value = float(match.group(1))
                if unit_type == 'pound': return value * 16
                if unit_type == 'gram': return value * 0.035274
                return value
            except ValueError: continue
    return np.nan

def clean_text(text):
    text = BeautifulSoup(str(text), "html.parser").get_text()
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    return text

# --- 3. Apply the functions to the DataFrame ---
print("Starting preprocessing...")
train_df['ipq'] = train_df['catalog_content'].apply(extract_ipq)
train_df['weight_oz'] = train_df['catalog_content'].apply(extract_weight_volume)
train_df['cleaned_content'] = train_df['catalog_content'].apply(clean_text)
print("Preprocessing complete!")

# --- 4. CRITICAL: Save the result to a new file ---
train_df.to_csv('data/train_processed.csv', index=False)
print("\nSuccessfully created and saved 'data/train_processed.csv'!")

Starting preprocessing...


C:\Users\shash\AppData\Local\Temp\ipykernel_6020\3990219536.py:51: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(str(text), "html.parser").get_text()


Preprocessing complete!

Successfully created and saved 'data/train_processed.csv'!
